<a href="https://colab.research.google.com/github/GlaydsonCavalcante/Monitoramento_Mercado_Busca_V2/blob/main/Monitoramento_Mercado_Busca_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧭 Como Configurar Suas Pesquisas (Guia Rápido)

Imagine que a variável `MONITORAMENTOS` é como uma **estante cheia de caixas organizadoras**:
1. **O Nome da Caixa (O Tema):** É o assunto geral que você quer acompanhar (exemplo: `"Inteligência Artificial"`). Fica sempre do lado esquerdo antes dos dois pontos `:`.
2. **O que tem dentro da Caixa (As Palavras de Busca):** É a listinha de frases que o robô vai procurar na internet para aquele tema. Fica sempre entre colchetes `[` e `]`.

---

### ⚠️ 3 Regrinhas de Ouro para não travar o robô:
* **Aspas Duplas e Simples:** Quando quiser que o robô busque palavras grudadas na ordem exata, use aspas duplas dentro de aspas simples: `'"inteligência artificial" bancos'`.
* **Atenção às Vírgulas:** Cada frase de busca precisa terminar com uma vírgula `,` para o robô saber que a frase acabou e vem outra em seguida.
* **Não apague as chaves `{ }` nem os colchetes `[ ]`:** Eles são as divisórias das caixas!

---

### 💡 Exemplos de como o robô entende:
* `'pix "banco central"'` $\rightarrow$ Vai buscar notícias que tenham a palavra **pix** e a expressão exata **banco central**.
* `'"fraude digital" biometria'` $\rightarrow$ Vai buscar notícias que tenham a expressão **fraude digital** junto com a palavra **biometria**.
* `'segurança -"futebol"'` $\rightarrow$ O sinal de menos `-` diz: "pegue notícias de segurança, mas jogue fora se falar de futebol".

In [ ]:
"""Configurações e parâmetros gerais de monitoramento."""

# PAÍSES DE EMISSÃO (Região geográfica da fonte/veículo de notícia - ISO 3166-1 alpha-2)
# BR: Brasil (Base)
# US: EUA (Hegemonia atual/Tech)
# GB: Reino Unido (Finanças/Geopolítica)
# PT: Portugal (Europa/Pontes lusófonas)
# ES: Espanha (Europa/Pontes hispânicas)
# CN: China (Superpotência/Tech)
# IN: Índia (Demografia/Crescimento)
# DE: Alemanha (Líder Europeu/Energia)
# JP: Japão (Automação/Envelhecimento)
# AU: Austrália (Minerais críticos/Transição)
# ID: Indonésia (Sudeste Asiático/Recursos)
# NG: Nigéria (Futuro demográfico da África)

PAISES_EMISSAO = ["BR", "US", "GB", "PT", "ES", "CN", "IN", "DE", "JP", "AU", "ID", "NG"]

# IDIOMAS (Língua em que a matéria foi redigida - BCP 47)
# Nota: Corrigido "en-UK" para o padrão oficial "en-GB"
IDIOMAS = [
    "pt-BR", "pt-PT",
    "en-US", "en-GB", "en-IN", "en-NG", "en-AU",
    "es-ES",
    "zh-CN",
    "hi",
    "de",
    "ja",
    "id"
]

# JANELA TEMPORAL DE BUSCA (Operador when: do Google Notícias), Opções válidas:
#   "1d"  -> Últimas 24 horas               "7d"  -> Últimos 7 dias (Recomendado para rotina semanal)
#   "30d" -> Últimos 30 dias (Último mês)   "90d" -> Últimos 3 meses                "1y"  -> Último ano
PERIODO_BUSCA = "1d"

# TERMOS EXCLUÍDOS (Palavras para descartar ruídos e notícias irrelevantes):
TERMOS_EXCLUIDOS = ["esporte", "futebol", "celebridade", "horóscopo", "receita"]

# FONTES / DOMÍNIOS PREFERENCIAIS (Opcional - deixe vazio [] para pesquisar em toda a web):
# Exemplo: ["valor.globo.com", "infomoney.com.br", "finsidersbrasil.com.br"]
DOMINIOS_PREFERENCIAIS = []

# SENSIBILIDADE DE DESDUPLICAÇÃO SEMÂNTICA:
# Valor entre 0.0 e 1.0 (quanto maior, mais parecidos os títulos precisam ser para agrupar)
SIMILARIDADE_REDUNDANCIA = 0.88

# ESTRUTURA DE MONITORAMENTO (Temas e termos de busca):
MONITORAMENTOS = {
    "Comércio Agentico": [
        '"inteligência artificial" bancos',
        '"IA como Orquestradora da Jornada de Compra"',
        '"IA autônoma"',
        'Phygital Omnicanalidade',
        '"Jornadas hiperpersonalizadas"',
        '"Experiências imersivas" "Social Commerce"'
    ],
    "Longevidade": [
        '"reconfiguração demográfica"',
        '"Envelhecimento da população"',
        '"Pacto intergeracional"',
        '"Saúde digital" "medicina personalizada"',
        '"Longevidade ativa" bem-estar "cuidado contínuo"',
        '"Terapias avançadas" "bioengenharia genética"',
    ],
}

# Funções de Resolução de URLs, NLP e Extração de Texto

In [ ]:
# Instalação das bibliotecas necessárias para RSS, decodificação de URLs, NLP leve e extração de texto
# Célula 1: Instalação das bibliotecas necessárias
!pip install --quiet requests pandas trafilatura rapidfuzz sentence-transformers googlenewsdecoder beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.0 MB/s eta 0:00:00


In [ ]:
# Célula 3: Funções de busca, desduplicação, resolução determinística de URLs e extração
import base64
from datetime import datetime
import json
import os
import random
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from googlenewsdecoder import new_decoderv1
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
import requests
from sentence_transformers import SentenceTransformer
import trafilatura

# Rotação de User-Agents modernos
USER_AGENTS = [
    (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    ),
    (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like"
        " Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
]


def build_rss_query(
    base_term: str,
    excluded_terms: list,
    period: str,
    preferred_domains: list,
) -> str:
    """Monta a query combinando termo, período, fontes e termos excluídos."""
    parts = [base_term]
    if period:
        parts.append(f"when:{period}")
    if preferred_domains:
        sites_query = " OR ".join(
            [f"site:{domain}" for domain in preferred_domains]
        )
        parts.append(f"({sites_query})")
    if excluded_terms:
        parts.extend([f"-{term}" for term in excluded_terms])
    return " ".join(parts)


def fetch_rss_feed(query: str, hl: str, gl: str) -> list:
    """Consulta o Google News RSS e extrai metadados dos artigos."""
    encoded_query = urllib.parse.quote(query)
    ceid = f"{hl.upper()}:{gl.upper()}"
    url = f"https://news.google.com/rss/search?q={encoded_query}&hl={hl}&gl={gl}&ceid={ceid}"

    headers = {"User-Agent": random.choice(USER_AGENTS)}
    try:
        response = requests.get(url, headers=headers, timeout=12)
        if response.status_code != 200:
            return []

        root = ET.fromstring(response.content)
        feed_items = []

        for item in root.findall(".//channel/item"):
            title = (
                item.find("title").text
                if item.find("title") is not None
                else ""
            )
            link = (
                item.find("link").text if item.find("link") is not None else ""
            )
            pub_date = (
                item.find("pubDate").text
                if item.find("pubDate") is not None
                else ""
            )
            source = (
                item.find("source").text
                if item.find("source") is not None
                else "Google News"
            )
            snippet = (
                item.find("description").text
                if item.find("description") is not None
                else ""
            )

            feed_items.append({
                "titulo": title.strip(),
                "link": link.strip(),
                "data_noticia": pub_date.strip(),
                "fonte": source.strip(),
                "snippet": snippet.strip(),
            })
        return feed_items
    except Exception:
        return []


def cluster_articles(
    raw_articles: list, similarity_threshold: float = 0.88
) -> list:
    """Agrupa notícias similares em clusters via embeddings multilingues e fuzzy matching."""
    if not raw_articles:
        return []

    print("Carregando modelo de similaridade semântica para agrupamento...")
    model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    titles = [a["titulo"] for a in raw_articles]
    embeddings = model.encode(
        titles, convert_to_numpy=True, normalize_embeddings=True
    )

    clusters = []
    visited = set()

    for i in range(len(raw_articles)):
        if i in visited:
            continue

        cluster_members = [raw_articles[i]]
        visited.add(i)

        for j in range(i + 1, len(raw_articles)):
            if j in visited:
                continue

            fuzzy_sim = fuzz.token_set_ratio(titles[i], titles[j]) / 100.0
            cosine_sim = float(np.dot(embeddings[i], embeddings[j]))

            # Critério duplo para evitar agrupar notícias distintas sobre o mesmo tema geral
            if (
                cosine_sim >= similarity_threshold and fuzzy_sim >= 0.80
            ) or fuzzy_sim >= 0.92:
                cluster_members.append(raw_articles[j])
                visited.add(j)

        primary = cluster_members[0]
        # Limita espelhos a no máximo 2 para evitar sobrecarga de requisições
        mirrors = [
            m["link"]
            for m in cluster_members[1:3]
            if m["link"] != primary["link"]
        ]

        clusters.append({
            "id_cluster": (
                f"CLUS_{datetime.now().strftime('%Y%m%d')}_{len(clusters) + 1:04d}"
            ),
            "tema": primary["tema"],
            "termo_origem": primary["termo_origem"],
            "titulo": primary["titulo"],
            "data_noticia": primary["data_noticia"],
            "fonte_principal": primary["fonte"],
            "url_primaria": primary["link"],
            "urls_espelho": mirrors,
            "snippet": primary["snippet"],
            "pais_emissao": primary["pais_emissao"],
            "idioma": primary["idioma"],
        })
    return clusters


def decode_token_offline(token: str) -> str:
    """Extrai a URL real diretamente dos bytes do protobuf em base64 (sem requisições externas)."""
    try:
        padded = token + "=" * (-len(token) % 4)
        raw_bytes = base64.urlsafe_b64decode(padded)
        matches = re.findall(
            rb"https?://[a-zA-Z0-9_\-\.\/\?\=\&\%\#\:\@]+", raw_bytes
        )
        for url_bytes in matches:
            url_str = url_bytes.decode("utf-8", errors="ignore")
            if (
                "google.com" not in url_str
                and "schema.org" not in url_str
                and len(url_str) > 15
            ):
                return url_str
    except Exception:
        pass
    return None


def resolve_publisher_url(google_news_url: str) -> str:
    """Resolve a URL direta do portal com decodificação offline e fallbacks seguros."""
    if not google_news_url or "news.google.com" not in google_news_url:
        return google_news_url

    clean_url = google_news_url.split("?")[0].strip()
    match = re.search(r"/articles/([^/?&]+)", clean_url)
    token = match.group(1) if match else None

    # 1. Tentativa offline direta (imune a rate limit)
    if token:
        extracted = decode_token_offline(token)
        if extracted:
            return extracted

    # 2. Tentativa via googlenewsdecoder
    try:
        res = new_decoderv1(google_news_url)
        if res.get("status") and res.get("decoded_url"):
            decoded = res["decoded_url"]
            if decoded.startswith("http") and "news.google.com" not in decoded:
                return decoded
    except Exception:
        pass

    # 3. Fallback via requisição direta ao link
    try:
        headers = {"User-Agent": random.choice(USER_AGENTS)}
        resp = requests.get(google_news_url, headers=headers, timeout=6)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            canonical = soup.find("link", rel="canonical")
            if (
                canonical
                and canonical.get("href")
                and "google.com" not in canonical["href"]
            ):
                return canonical["href"]
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if href.startswith("http") and "google.com" not in href:
                    return href
    except Exception:
        pass

    return google_news_url


def scrape_article_text(url: str) -> tuple:
    """Extrai o texto integral diretamente no domínio do veículo jornalístico."""
    real_url = resolve_publisher_url(url)

    # Se a URL não foi decodificada e manteve o link do Google, aborta para não ler página vazia
    if "news.google.com" in real_url:
        return None, "FALHA_DECODIFICACAO_URL", real_url

    try:
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
            ),
            "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7,es;q=0.6",
        }
        html_content = None

        resp = requests.get(real_url, headers=headers, timeout=12)
        if resp.status_code == 200:
            html_content = resp.text
        else:
            return None, f"FALHA_HTTP_{resp.status_code}", real_url

        if not html_content:
            return None, "FALHA_CONTEUDO_VAZIO", real_url

        # Extração primária com trafilatura
        text = trafilatura.extract(
            html_content,
            include_comments=False,
            include_tables=False,
            include_links=False,
            output_format="txt",
        )

        # Fallback com BeautifulSoup para portais com markup específico
        if not text or len(text.strip()) < 150:
            soup = BeautifulSoup(html_content, "html.parser")
            for tag in soup(
                ["script", "style", "nav", "header", "footer", "aside", "form"]
            ):
                tag.decompose()
            paragraphs = [
                p.get_text().strip()
                for p in soup.find_all("p")
                if len(p.get_text().strip()) > 35
            ]
            text = "\n\n".join(paragraphs)

        if text and len(text.strip()) > 150:
            return text.strip(), "SUCESSO", real_url
        return None, "CONTEUDO_INSUFICIENTE", real_url

    except Exception as exc:
        return None, f"ERRO_EXCEPTION: {str(exc)}", real_url


def process_cluster_with_fallback(cluster: dict) -> dict:
    """Processa o cluster decodificando URLs e gravando links finais dos publishers."""
    urls_to_try = [cluster["url_primaria"]] + cluster["urls_espelho"]
    historico = []

    # Decodifica previamente todos os links espelho
    espelhos_decodificados = [
        resolve_publisher_url(espelho) for espelho in cluster["urls_espelho"]
    ]

    primeira_url_decodificada = None

    for link in urls_to_try:
        texto, status, final_url = scrape_article_text(link)

        if primeira_url_decodificada is None:
            primeira_url_decodificada = final_url

        historico.append({
            "url_original_rss": link,
            "url_canonica_decodificada": final_url,
            "status": status,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

        if status == "SUCESSO":
            return {
                "id_cluster": cluster["id_cluster"],
                "tema": cluster["tema"],
                "termo_origem": cluster["termo_origem"],
                "titulo": cluster["titulo"],
                "data_noticia": cluster["data_noticia"],
                "idioma": cluster["idioma"],
                "fonte_utilizada": cluster["fonte_principal"],
                "url_utilizada": final_url,
                "urls_espelho_disponiveis": espelhos_decodificados,
                "status_extracao": "SUCESSO",
                "texto_completo": texto,
                "motivo_bloqueio": None,
                "necessita_extracao_manual": False,
                "historico_tentativas": historico,
            }

    # Em caso de bloqueio, garante a URL canônica decodificada registrada
    return {
        "id_cluster": cluster["id_cluster"],
        "tema": cluster["tema"],
        "termo_origem": cluster["termo_origem"],
        "titulo": cluster["titulo"],
        "data_noticia": cluster["data_noticia"],
        "idioma": cluster["idioma"],
        "fonte_utilizada": cluster["fonte_principal"],
        "url_utilizada": (
            primeira_url_decodificada
            if primeira_url_decodificada
            else cluster["url_primaria"]
        ),
        "urls_espelho_disponiveis": espelhos_decodificados,
        "status_extracao": "BLOQUEADO",
        "texto_completo": "[CONTEUDO_BLOQUEADO]",
        "motivo_bloqueio": (
            f"Acesso protegido ou indisponível em todas as {len(urls_to_try)}"
            " fontes testadas."
        ),
        "necessita_extracao_manual": True,
        "historico_tentativas": historico,
    }

# Execução do Motor e Exportação

In [ ]:
import concurrent.futures
from datetime import datetime
import json
import os
import random
import time
import requests

RAW_LAKE_FILE = "lake_raw_noticias.json"
CHECKPOINT_FILE = "checkpoint_processamento.json"
FINAL_OUTPUT_JSON = (
    f"noticias_processadas_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

MAX_WORKERS_PARALELO = globals().get("MAX_WORKERS_PARALELO", 4)


def enviar_telegram(caminho_arquivo: str, total_noticias: int, sucessos: int) -> None:
    """Envia o arquivo consolidado para o canal ou chat de distribuição."""
    bot_token = os.getenv("TELEGRAM_BOT_TOKEN")
    chat_id = os.getenv("TELEGRAM_CHAT_ID")

    if not bot_token or not chat_id:
        print("Credenciais do Telegram não configuradas no ambiente.")
        return

    url = f"https://api.telegram.org/bot{bot_token}/sendDocument"
    legenda = (
        f"📊 *Relatório de Monitoramento Concluído*\n"
        f"• Total de Notícias Únicas: {total_noticias}\n"
        f"• Textos Extraídos: {sucessos}\n"
        f"• Bloqueadas/Manuais: {total_noticias - sucessos}\n"
        f"• Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}"
    )

    with open(caminho_arquivo, "rb") as doc:
        payload = {
            "chat_id": chat_id,
            "caption": legenda,
            "parse_mode": "Markdown",
        }
        files = {"document": doc}
        resp = requests.post(url, data=payload, files=files, timeout=60)
        resp.raise_for_status()
    print("Relatório enviado com sucesso via Telegram.")


# 1. Coleta RSS
raw_articles = []
if os.path.exists(RAW_LAKE_FILE):
    with open(RAW_LAKE_FILE, "r", encoding="utf-8") as f:
        raw_articles = json.load(f)
else:
    for tema, termos in MONITORAMENTOS.items():
        for termo in termos:
            for gl in PAISES_EMISSAO:
                for hl in IDIOMAS:
                    q = build_rss_query(
                        termo,
                        TERMOS_EXCLUIDOS,
                        PERIODO_BUSCA,
                        DOMINIOS_PREFERENCIAIS,
                    )
                    itens = fetch_rss_feed(q, hl=hl, gl=gl)
                    for item in itens:
                        item["tema"] = tema
                        item["termo_origem"] = termo
                        item["pais_emissao"] = gl
                        item["idioma"] = hl
                        raw_articles.append(item)

    with open(RAW_LAKE_FILE, "w", encoding="utf-8") as f:
        json.dump(raw_articles, f, ensure_ascii=False, indent=2)

# 2. Deduplicação Semântica
clusters = cluster_articles(
    raw_articles,
    similarity_threshold=globals().get("SIMILARIDADE_REDUNDANCIA", 0.88),
)

# 3. Extração Multithread
checkpoint_data = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint_data = json.load(f)

clusters_pendentes = [
    c for c in clusters if c["id_cluster"] not in checkpoint_data
]


def worker_extracao(cluster_item):
    time.sleep(random.uniform(0.4, 1.0))
    return process_cluster_with_fallback(cluster_item)


with concurrent.futures.ThreadPoolExecutor(
    max_workers=MAX_WORKERS_PARALELO
) as executor:
    futuros = {
        executor.submit(worker_extracao, c): c for c in clusters_pendentes
    }

    for i, futuro in enumerate(concurrent.futures.as_completed(futuros), 1):
        resultado = futuro.result()
        checkpoint_data[resultado["id_cluster"]] = resultado
        if i % 10 == 0 or i == len(clusters_pendentes):
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(checkpoint_data, f, ensure_ascii=False, indent=2)

# 4. Geração do JSON Final e Despacho
processed_results = list(checkpoint_data.values())
with open(FINAL_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(processed_results, f, ensure_ascii=False, indent=2)

sucessos = sum(1 for r in processed_results if r["status_extracao"] == "SUCESSO")
enviar_telegram(FINAL_OUTPUT_JSON, len(processed_results), sucessos)

Carregando registros brutos de 'lake_raw_noticias.json'...
Carregando modelo de similaridade semântica para agrupamento...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Deduplicação concluída: 2117 matérias agrupadas em 117 clusters únicos.

Iniciando extração paralela (117 pendentes de 117 totais)...
[1/117] ✅ Febraban Tech 2026: Bancos usam IA, mas decisões sensíveis f...
[2/117] ✅ Febraban Tech 2026: Inteligência artificial mudou a maneira ...
[3/117] ✅ IA nos bancos: da experiência do cliente à rotina da equipe,...
[4/117] ✅ IA muda atendimento, crédito e rotina de funcionários nos ma...
[5/117] ✅ Se uma IA do banco errar e causar prejuízo ao cliente, quem ...


[6/117] 🔒 FEBRABAN TECH | Depois de anos de investimentos em tecnologi...
[7/117] ✅ Banco do Brasil lança novo aplicativo com inteligência artif...
[8/117] ✅ Banco do Brasil lança novo aplicativo com foco em IA e perso...
[9/117] ✅ Caixa saiu da 'selva analógica' com a IA em 3 anos - Mobile ...
[10/117] ✅ Banco Do Brasil Reformula Aplicativo E Amplia Uso De Intelig...
[11/117] ✅ Banco do Brasil muda aplicativo e aposta em IA para personal...
[12/117] ✅ Quanto a IA já avançou no dia a dia dos grandes bancos do Pa...
[13/117] ✅ Febraban Tech: Bancos puxam investimentos em IA no Brasil, d...
[14/117] ✅ Na Febraban Tech da Inteligência Artificial, Banco Central d...
[15/117] ✅ Banco do Brasil lança novo app bb com foco em inteligencia a...
[16/117] ✅ Bradesco projeta era do “banco conversacional” impulsionada ...
[17/117] ✅ Banco do Brasil lança novo app com inteligência artificial e...
[18/117] ✅ Estudo mostra que, na era da IA, confiança é o atributo mais...
[19/117] ✅ Banco do Brasil ap

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>